In [ ]:
import cv2
import mediapipe as mp
import warnings
warnings.filterwarnings("ignore")


In [ ]:
mp_hands = mp.solutions.hands
hands = mp_hands.Hands()  
mp_draw = mp.solutions.drawing_utils 


cap = cv2.VideoCapture(0)

cap.set(3, 2000) 
cap.set(4, 2000)


finger_tips = [8, 12, 16, 20]  # Indices of fingertip landmarks
thumb_tip = 4  # Index of thumb tip landmark


like_img = cv2.imread(r"D:\Computer Vision\Real-Time Finger Counting with Computer Vision\images\like.png")  
dislike_img = cv2.imread(r"D:\Computer Vision\Real-Time Finger Counting with Computer Vision\images\dislike.png") 


while True:
    ret, img = cap.read()
    
    if ret == False:
        break

    img = cv2.flip(img, 1)  # Flip the image horizontally 180
    h, w, c = img.shape  # Get image dimensions

    results = hands.process(img)  # Process the image for hand detection

    if results.multi_hand_landmarks:  # If hands are detected
        for hand in results.multi_hand_landmarks:  # Iterate through each detected hand
            lm_list = []  # Store landmark coordinates
            for id, lm in enumerate(hand.landmark):
                lm_list.append(lm)

            # Check for finger folds
            finger_fold_status = []
            for tip in finger_tips:
                x, y = int(lm_list[tip].x * w), int(lm_list[tip].y * h)
                cv2.circle(img, (x, y), 15, (255, 0, 0), cv2.FILLED)

                if lm_list[tip].x < lm_list[tip - 3].x:  # If fingertip is bent
                    cv2.circle(img, (x, y), 15, (255, 0, 0), cv2.FILLED)  # Draw green circles on bent fingertips
                    finger_fold_status.append(True)
                else:
                    finger_fold_status.append(False)
            
            
            # stop 
            if lm_list[4].y < lm_list[2].y and lm_list[8].y < lm_list[6].y and lm_list[12].y < lm_list[10].y and\
                 lm_list[16].y < lm_list[14].y and lm_list[20].y < lm_list[18].y and lm_list[17].x < lm_list[0].x  < lm_list[5].x:
                
                    cv2.putText(img, "STOP", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0,255), 3)
                    print("STOP")
                
             # Forward       
            if lm_list[3].x < lm_list[4].x and lm_list[8].y < lm_list[6].y and lm_list[12].y > lm_list[10].y and\
                 lm_list[16].y > lm_list[14].y and lm_list[20].y > lm_list[18].y :
                
                    cv2.putText(img, "FROWARD", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0,255), 3)
                    print("FROWARD")
                
                
             # Backward       
            if lm_list[3].x > lm_list[4].x and lm_list[3].y < lm_list[4].y and lm_list[8].y > lm_list[6].y and\
                 lm_list[12].y < lm_list[16].y and lm_list[20].y < lm_list[18].y :
                
                    cv2.putText(img, "BACKWARD", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0,255), 3)
                    print("BACKWARD")
                
        
            # LEFT    
            if lm_list[4].y < lm_list[2].y and lm_list[8].x < lm_list[6].x and lm_list[12].x > lm_list[10].x and\
                 lm_list[16].x > lm_list[14].x and lm_list[20].x > lm_list[18].x and lm_list[17].x < lm_list[0].x  < lm_list[5].x:
                
                    cv2.putText(img, "LEFT", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0,255), 3)
                    print("LEFT")
                
                
            # RIGHT    
            if lm_list[4].y < lm_list[2].y and lm_list[8].x > lm_list[6].x and lm_list[12].x < lm_list[10].x and\
                 lm_list[16].x < lm_list[14].x and lm_list[20].x < lm_list[18].x :
                
                    cv2.putText(img, "RIGHT", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0,255), 3)
                    print("RIGHT")



                # Check for like or dislike gesture
            if all(finger_fold_status):  # If all fingers are bent
                    if lm_list[thumb_tip].y < lm_list[thumb_tip - 1].y < lm_list[thumb_tip - 2].y:  # Like gesture
                        print("LIKE")
                        cv2.putText(img, "LIKE", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
                        h, w, c = like_img.shape
                        img[35:h + 35, 30:w + 30] = like_img  # Overlay like image
                    else:  # Dislike gesture
                        cv2.putText(img, "DISLIKE", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
                        h, w, c = dislike_img.shape
                        img[35:h + 35, 30:w + 30] = dislike_img  # Overlay dislike image

            mp_draw.draw_landmarks(img, hand, mp_hands.HAND_CONNECTIONS,  # Draw hand landmarks
                                  mp_draw.DrawingSpec((0, 0, 255), 6, 3),
                                  mp_draw.DrawingSpec((0, 255, 0), 4, 2))

    cv2.imshow("IMAGE", img)  
    if cv2.waitKey(1) == ord("q"):  
        break

cap.release()
cv2.destroyAllWindows()


: 